# DX 601 Week 3 Homework

## Introduction

In this homework, you will practice plotting data and calculating model predictions and losses.

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for sample code.

* https://github.com/bu-cds-omds/dx500-examples
* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Instructions

You should replace every instance of "..." or "TODO" below.
These are where you are expected to write code to answer each problem.

After some of the problems, there are extra code cells that will test functions that you wrote so you can quickly see how they run on an example.
If your code works on these examples, it is more likely to be correct.
However, the autograder will test different examples, so working correctly on these examples does not guarantee full credit for the problem.
You may change the example inputs to further test your functions on your own.
You may also add your own example inputs for problems where we did not provide any.

Be sure to run each code block after you edit it to make sure it runs as expected.
When you are done, we strongly recommend you run all the code from scratch (Runtime menu -> Restart and Run all) to make sure your current code works for all problems.

If your code raises an exception when run from scratch, it will  interfere with the auto-grader process causing you to lose some or all points for this homework.
Please ask for help in YellowDig or schedule an appointment with a learning facilitator if you get stuck.


## Shared Imports

Do not install or use any additional modules.
Installing additional modules may result in an autograder failure resulting in zero points for some or all problems.

In [3]:
%pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 39.8 MB/s  0:00:006m0:00:01

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
from collections import defaultdict
from enum import Enum
from typing import Dict, List, Tuple, Type
from pprint import pprint
import csv
import os
import numpy as np

# The Data

In this assignment you will be training a probabilistic model to predict whether an avocado is good to eat (or not) given a feature description of the avocado.


### Problem 1: Storing Conditional Probability Mass Functions

The code below creates some enums for the features we will be using in this homework. Please do not change this


In [5]:
class Color(Enum):
    BLACK = 0
    BROWN = 1
    GREEN = 2

    @classmethod
    def value_of(cls,
                 s: str
                 ) -> "Color":
        for x in cls._member_names_:
            if s.upper() == x:
                return cls.__dict__[x]
        raise ValueError(f"ERROR: unknown string {s}")



class Softness(Enum):
    MUSHY = 0
    SOFT = 1
    TENDER = 2
    HARD = 3

    @classmethod
    def value_of(cls,
                 s: str
                 ) -> "Softness":
        for x in cls._member_names_:
            if s.upper() == x:
                return cls.__dict__[x]
        raise ValueError(f"ERROR: unknown string {s}")


class GoodToEat(Enum):
    YES = 0
    NO = 1

    @classmethod
    def value_of(cls,
                 s: str
                 ) -> "GoodToEat":
        for x in cls._member_names_:
            if s.upper() == x:
                return cls.__dict__[x]
        raise ValueError(f"ERROR: unknown string {s}")

Below is some code to load the accopanying data file. Please do not change this

In [6]:
def load_data() -> List[Tuple[Color, Softness, GoodToEat]]:
    data_file: str = os.path.join("train_avacados.txt")
    if not os.path.exists(data_file):
        raise Exception(f"ERROR: file {data_file} does not exist!")

    data: List[Tuple[Color, Softness, GoodToEat]] = list()
    with open(data_file, "r") as f:
        reader = csv.reader(f, delimiter=",")

        for line in reader:
            if len(line) > 0:

                if len(line) != 3:
                    raise ValueError(f"ERROR: expected three values but got {line}")

                color, softness, good_to_eat = line
                data.append(tuple([Color.value_of(color.strip().rstrip()),
                                   Softness.value_of(softness.strip().rstrip()),
                                   GoodToEat.value_of(good_to_eat.strip().rstrip())]))

    return data

In [7]:
data = load_data()
pprint(data)

[(<Color.BLACK: 0>, <Softness.MUSHY: 0>, <GoodToEat.NO: 1>),
 (<Color.BLACK: 0>, <Softness.MUSHY: 0>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <Softness.MUSHY: 0>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.BROWN: 1>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.BROWN: 1>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.BROWN: 1>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.TENDER: 2>, <GoodToEat.YES: 0>),
 (<Color.BLACK: 0>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.HARD: 3>, <GoodToEat.NO: 1>),
 (<Color.GREEN: 2>, <Softness.SOFT: 1>, <GoodToEat.YES: 0>),
 (<Color.GREEN: 2>, <Softness.HARD: 3>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <Softness.HARD: 3>, <GoodToEat.NO: 1>),
 (<Color.GREEN: 2>, <Softness.TENDER: 2>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <Softness.TENDER: 2>, <GoodToEat.YES: 0>),
 (<Color.BLACK: 0>, <Softness.SOFT: 1>, <GoodToEat.NO: 1>),
 (<Color.BROWN: 1>, <S

Below is the beginning of a class. Complete the `fit` method to populate the three fields in this class from the training data. The field `good_to_eat_prior` should contain a probability mass function over the `GoodToEat` feature. You should calculate it as follows:

$$Pr[GoodToEat=a] = \dfrac{\#\text{ of times}(GoodToEat=a)}{\sum\limits_{b\in GoodToEat} \#\text{ of times}(GoodToEat=b)} $$

This field is a dictionary where the key is the value of `GoodToEat` and the value of the dictionary is the probability.

The field `softness_given_good_to_eat_pmf` should contain the following conditional probability distribution:
$Pr[Softness | GoodToEat]$. This distribution is stored as a dictionary of dictionaries. A key to the outer dictionary is a value of `GoodToEat` and using this key returns a pmf over the `Softness` feature. Therefore if you wanted to know a particular value
$Pr[Softness=a | GoodToEat=b]$ you would look it up in this field with the code `self.softness_given_good_to_eat_pmf[b][a]`. You should calculate these pmfs as follows:

$$Pr[Softness = a | GoodToEat = b]= \dfrac{\#\text{ of times}(Softness = a \cap GoodToEat=b)}{\#\text{ of times}(GoodToEat=b)}$$

Likewise you should follow a similar procedure to populate the `color_give_good_to_eat_pmf` field

In [13]:
# YOUR CHANGES HERE

class AvocadoPredictor(object):
    def __init__(self) -> None:
        self.color_given_good_to_eat_pmf: Dict[GoodToEat, Dict[Color, float]] = defaultdict(lambda: defaultdict(float))
        self.softness_given_good_to_eat_pmf: Dict[GoodToEat, Dict[Softness, float]] = defaultdict(lambda: defaultdict(float))
        self.good_to_eat_prior: Dict[GoodToEat, float] = defaultdict(float)

    """
    Problem 1: populate the fields from the constructor with the provided training data
    """
    def fit(self,
            data: List[Tuple[Color, Softness, GoodToEat]]
            ) -> "AvocadoPredictor":
        
        total = len(data)

        for color, softness, good_to_eat in data:
            self.good_to_eat_prior[good_to_eat] += 1
            self.color_given_good_to_eat_pmf[good_to_eat][color] += 1
            self.softness_given_good_to_eat_pmf[good_to_eat][softness] += 1

        for good_to_eat in self.good_to_eat_prior:
            count = self.good_to_eat_prior[good_to_eat]
            self.good_to_eat_prior[good_to_eat] /= total

            for color in self.color_given_good_to_eat_pmf[good_to_eat]:
                self.color_given_good_to_eat_pmf[good_to_eat][color] /= count

            for softness in self.softness_given_good_to_eat_pmf[good_to_eat]:
                self.softness_given_good_to_eat_pmf[good_to_eat][softness] /= count

        return self

    """
    Problem 2: use the fields set when the 'fit' method was called to estimate some GoodToEat values
    """
    def predict_color_proba(self,
                            X: List[Color]
                            ) -> List[List[Tuple[GoodToEat, float]]]:

        probs_per_example: List[List[Tuple[GoodToEat, float]]] = list()

        for color in X:
            pmf = []

            for good_to_eat in self.good_to_eat_prior:
                probability = (
                    self.color_given_good_to_eat_pmf[good_to_eat][color]
                    * self.good_to_eat_prior[good_to_eat]
                )
                pmf.append((good_to_eat, probability))

            total = sum(probability for _, probability in pmf)

            pmf = [
                (good_to_eat, probability / total)
                for good_to_eat, probability in pmf
            ]

            probs_per_example.append(pmf)

        return probs_per_example

    """
    Problem 2: use the fields set when the 'fit' method was called to estimate some GoodtoEat values
    """
    def predict_softness_proba(self,
                               X: List[Softness]
                               ) -> List[List[Tuple[GoodToEat, float]]]:

        probs_per_example: List[List[Tuple[GoodToEat, float]]] = list()

        for softness in X:
            pmf = []

            for good_to_eat in self.good_to_eat_prior:
                probability = (
                    self.softness_given_good_to_eat_pmf[good_to_eat][softness]
                    * self.good_to_eat_prior[good_to_eat]
                )
                pmf.append((good_to_eat, probability))

            total = sum(probability for _, probability in pmf)

            pmf = [
                (good_to_eat, probability / total)
                for good_to_eat, probability in pmf
            ]

            probs_per_example.append(pmf)

        return probs_per_example

    def predict_color(self,
                      X: List[Color]
                      ) -> List[GoodToEat]:
        predictions: List[GoodToEat] = list()

        for pmf in self.predict_color_proba(X):
            predictions.append(max(pmf, key=lambda x: x[1])[0])

        return predictions

    def predict_softness(self,
                         X: List[Softness]
                         ) -> List[GoodToEat]:
        predictions: List[GoodToEat] = list()

        for pmf in self.predict_softness_proba(X):
            predictions.append(max(pmf, key=lambda x: x[1])[0])

        return predictions

Check that some of your values are correct

In [14]:
m = AvocadoPredictor().fit(data)

pprint(m.good_to_eat_prior)
assert(np.isclose(m.good_to_eat_prior[GoodToEat.YES], 0.52941))
assert(np.isclose(m.good_to_eat_prior[GoodToEat.NO], 0.47059))

pprint(m.color_given_good_to_eat_pmf)
assert(np.isclose(m.color_given_good_to_eat_pmf[GoodToEat.YES][Color.BLACK], 0.111111))
assert(np.isclose(m.color_given_good_to_eat_pmf[GoodToEat.NO][Color.BLACK], 0.375))

pprint(m.softness_given_good_to_eat_pmf)
assert(np.isclose(m.softness_given_good_to_eat_pmf[GoodToEat.YES][Softness.MUSHY], 0.111111))
assert(np.isclose(m.softness_given_good_to_eat_pmf[GoodToEat.NO][Softness.MUSHY], 0.375))

defaultdict(<class 'float'>,
            {<GoodToEat.YES: 0>: 0.5294117647058824,
             <GoodToEat.NO: 1>: 0.47058823529411764})
defaultdict(<function AvocadoPredictor.__init__.<locals>.<lambda> at 0x78e5d0e822a0>,
            {<GoodToEat.YES: 0>: defaultdict(<class 'float'>,
                                             {<Color.BROWN: 1>: 0.5555555555555556,
                                              <Color.BLACK: 0>: 0.1111111111111111,
                                              <Color.GREEN: 2>: 0.3333333333333333}),
             <GoodToEat.NO: 1>: defaultdict(<class 'float'>,
                                            {<Color.BROWN: 1>: 0.25,
                                             <Color.BLACK: 0>: 0.375,
                                             <Color.GREEN: 2>: 0.375})})
defaultdict(<function AvocadoPredictor.__init__.<locals>.<lambda> at 0x78e5d0e82350>,
            {<GoodToEat.YES: 0>: defaultdict(<class 'float'>,
                                         

### Problem 2: Predicting GoodToEat Given a Feature

Now complete the `predict_color_proba` and `predict_softness_proba` methods. These methods are symmetric (i.e. they behave the same way) except `predict_color_proba` processes the `Color` feature while `predict_softness_proba` processes the `Softness` feature (so don't forget to make that change when implementing the two)! Therefore I am only going to describe what I want from the `predict_color_proba` method. Given a `list` of `Color` features (one `Color` feature per avocado), I want you to return an entire pmf over the `GoodToEat` attribute (one pmf per avocado). If the input list has 10 `Color` values then your method should produce 10 pmfs. Each pmf will look like this:

$$[(GoodToEat.YES, Pr[GoodToEat=YES|Color=c]), (GoodToEat.NO, Pr[GoodToEat=No|Color=c])]$$

The way you should calculate the probability $Pr[GoodToEat=x | Color=c]$ is by using Bayes' rule:

$$Pr[GoodToEat=x | Color=c] = \dfrac{Pr[GoodToEat=x]Pr[Color=c|GoodToEat=x]}{Pr[Color=c]}$$

You will need to use the law of total probability to calculate $Pr[Color=c]$ as follows:

$$Pr[Color=c] = \sum\limits_{x\in GoodToEat} Pr[Color=c | GoodToEat=x]Pr[GoodToEat=x]$$

In [23]:
def predict_color_proba(self,
                        X: List[Color]
                        ) -> List[List[Tuple[GoodToEat, float]]]:

    probs_per_example = []

    for color in X:
        p_color = 0

        for good_to_eat in [GoodToEat.YES, GoodToEat.NO]:
            p_color += (
                self.color_given_good_to_eat_pmf[good_to_eat][color]
                * self.good_to_eat_prior[good_to_eat]
            )

        yes_probability = (
            self.color_given_good_to_eat_pmf[GoodToEat.YES][color]
            * self.good_to_eat_prior[GoodToEat.YES]
        ) / p_color

        no_probability = (
            self.color_given_good_to_eat_pmf[GoodToEat.NO][color]
            * self.good_to_eat_prior[GoodToEat.NO]
        ) / p_color

        probs_per_example.append([
            (GoodToEat.YES, yes_probability),
            (GoodToEat.NO, no_probability)
        ])

    return probs_per_example

In [24]:
def predict_softness_proba(self,
                           X: List[Softness]
                           ) -> List[List[Tuple[GoodToEat, float]]]:

    probs_per_example = []

    for softness in X:
        p_softness = 0

        for good_to_eat in [GoodToEat.YES, GoodToEat.NO]:
            p_softness += (
                self.softness_given_good_to_eat_pmf[good_to_eat][softness]
                * self.good_to_eat_prior[good_to_eat]
            )

        yes_probability = (
            self.softness_given_good_to_eat_pmf[GoodToEat.YES][softness]
            * self.good_to_eat_prior[GoodToEat.YES]
        ) / p_softness

        no_probability = (
            self.softness_given_good_to_eat_pmf[GoodToEat.NO][softness]
            * self.good_to_eat_prior[GoodToEat.NO]
        ) / p_softness

        probs_per_example.append([
            (GoodToEat.YES, yes_probability),
            (GoodToEat.NO, no_probability)
        ])

    return probs_per_example

In [26]:
m = AvocadoPredictor().fit(data)

black_prediction = m.predict_color_proba([Color.BLACK])
pprint(black_prediction)

assert(black_prediction[0][0][0] == GoodToEat.YES)
assert(black_prediction[0][1][0] == GoodToEat.NO)

soft_prediction = m.predict_softness_proba([Softness.SOFT])
pprint(soft_prediction)

assert(soft_prediction[0][0][0] == GoodToEat.YES)
assert(soft_prediction[0][1][0] == GoodToEat.NO)

[[(<GoodToEat.NO: 1>, 0.7499999999999999), (<GoodToEat.YES: 0>, 0.25)]]


AssertionError: 

### Problem 3: Making a Decision

Now complete the `predict_color` and `predict_softness` methods. These method are symmetric (i.e. they behave the same way) except `predict_color` processes the `Color` feature while `predict_softness` processes the `Softness` feature (so don't forget to make that change when implementing the two)! Therefore I am only going to describe what I want from the `predict_color` method. Given a `list` of `Color` features (one `Color` feature per avocado), I want you to return a single `GoodToEat` value. If the input list has 10 `Color` values then your method should produce 10 `GoodToEat` values. The `GoodToEat` value you should produce for a single avocado is the most likely `GoodToEat` value given the provided `Color` value. I would recommend calling `predict_color_proba` and than returning whichever `GoodToEat` value has the largest probability.

Below is a function which will evaluate the accuracy of your model

In [27]:
def accuracy(predictions: List[GoodToEat],
             actual: List[GoodToEat]
             ) -> float:
    if len(predictions) != len(actual):
        raise ValueError(f"ERROR: expected predictions and actual to be same length but got pred={len(predictions)}" +
            " and actual={len(actual)}")

    num_correct: float = 0
    for pred, act in zip(predictions, actual):
        num_correct += int(pred == act)

    return num_correct / len(predictions)

In [28]:
# YOUR CHANGES HERE
m = AvocadoPredictor().fit(data)

color_data: List[Color] = [color for color, _, _ in data]
softness_data: List[Softness] = [softness for _, softness, _ in data]
good_to_eat_data: List[GoodToEat] = [good_to_eat for _, _, good_to_eat in data]

print("Accuracy when predicting only on color: {:.4f}".format(accuracy(m.predict_color(color_data), good_to_eat_data)))
print("Accuracy when predicting only on softness: {:.4f}".format(accuracy(m.predict_softness(softness_data), good_to_eat_data)))

Accuracy when predicting only on color: 0.6471
Accuracy when predicting only on softness: 0.8235
